## Evaluation report visualization notebook

This notebook loads the merged evaluation workbook created by the report-generation step and visualizes each compatible sheet.

Each plotted sheet should contain these columns:

- `Name` — model or experiment name
- `TP`, `TN`, `FP`, `FN` — confusion-matrix counts
- `Accuracy` — evaluation accuracy, expected on a 0–1 scale

The notebook creates one chart per compatible sheet. Each chart shows confusion-matrix counts as grouped bars and accuracy as a green line on a secondary y-axis.

In [ ]:
import shutil
from contextlib import suppress
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## Configuration

Edit `ROOT_PATH` and `PROJECT_NAME` before running the notebook.

The derived paths are:

- `DATA_ROOT` — root data folder inside the repository
- `EVALUATION_DIR` — project-specific evaluation output folder
- `REPORT_DIR` — folder containing the merged report workbook
- `MERGED_REPORT_PATH` — final Excel workbook to load and visualize

Run this cell first and verify the printed paths point to the correct project before plotting.

In [ ]:
# Advanced users: edit these paths directly in this notebook.
ROOT_PATH = Path("sprout-emergence-prediction")
PROJECT_NAME = "nabila"

DATA_ROOT = ROOT_PATH / "data"
EVALUATION_DIR = DATA_ROOT / "evaluation" / PROJECT_NAME
REPORT_DIR = EVALUATION_DIR / "report"


TIME_STEP = 3
DATA_RANGE = 9
INPUT_CSV = (
    EVALUATION_DIR
    / f"Evaluation_results_ef2_tcn_{TIME_STEP}-{DATA_RANGE}_org_nab_relearned.csv"
)
MERGED_REPORT_PATH = REPORT_DIR / "MERGE_report.xlsx"

plt.style.use("default")

print("Repo root:", ROOT_PATH)
print("Project name:", PROJECT_NAME)
print("Evaluation dir:", EVALUATION_DIR)
print("Report dir:", REPORT_DIR)
print("Merged workbook:", MERGED_REPORT_PATH)

## Plotting helper functions

This section defines the reusable functions used for report visualization:

- `load_excel_sheets()` opens the merged workbook and loads all sheets into pandas DataFrames.
- `prepare_sheet_for_plotting()` verifies that a sheet has the required columns and sorts rows by `Accuracy`.
- `plot_excel_sheet()` draws one combined chart for a single sheet.
- `plot_merged_excel_report()` loops through all sheets and skips any sheet that does not match the expected schema.

Sheets with different column names are skipped instead of crashing the whole notebook. This is useful because the merged workbook may contain both standard evaluation sheets and specialized box-level sheets.

In [ ]:
PLOT_COLUMNS = ["Name", "TP", "TN", "FP", "FN", "Accuracy"]


def load_excel_sheets(file_path: Path) -> dict[str, pd.DataFrame]:
    """Load all sheets from an Excel workbook.

    Args:
        file_path: Path to the Excel workbook to load.

    Returns:
        Dictionary mapping sheet names to loaded dataframes.
    """
    excel_file = pd.ExcelFile(file_path)
    return {
        sheet_name: pd.read_excel(excel_file, sheet_name)
        for sheet_name in excel_file.sheet_names
    }


def prepare_sheet_for_plotting(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Sort and select columns used by the report plots.

    Args:
        dataframe: Report sheet dataframe containing confusion-matrix columns.

    Returns:
        Dataframe sorted by accuracy and limited to plot columns.

    Raises:
        ValueError: If any required plot column is missing.
    """
    missing_columns = [
        column for column in PLOT_COLUMNS if column not in dataframe.columns
    ]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    return dataframe.sort_values("Accuracy")[PLOT_COLUMNS]


def plot_excel_sheet(
    dataframe: pd.DataFrame,
    sheet_name: str,
    save_plots: bool = False,
) -> None:
    """Create and optionally save one confusion-matrix-plus-accuracy plot.

    Args:
        dataframe: Prepared report dataframe with model names and metrics.
        sheet_name: Name of the Excel sheet represented by the dataframe.
        save_plots: Whether to save the generated plot as a PNG file.
    """
    fig, ax1 = plt.subplots(figsize=(10, 8))

    dataframe.set_index("Name")[["TP", "TN", "FP", "FN"]].plot(
        kind="bar",
        ax=ax1,
    )
    ax1.set_title(f"Confusion Matrix Values and Accuracy for {sheet_name}")
    ax1.set_ylabel("Count")
    ax1.set_xlabel("Model architecture")
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90)
    ax1.yaxis.grid(visible=True)

    ax2 = ax1.twinx()
    ax2.plot(dataframe["Name"], dataframe["Accuracy"], color="green", marker="o")
    ax2.set_ylabel("Accuracy", color="green")
    ax2.set_ylim([0, 1])

    plt.tight_layout()
    if save_plots:
        fig.savefig(f"{sheet_name}_combined_plot.png")
    plt.show()


def plot_merged_excel_report(file_path: Path, save_plots: bool = False) -> None:
    """Plot all compatible sheets from the merged Excel report.

    Args:
        file_path: Path to the merged Excel report workbook.
        save_plots: Whether to save each generated plot as a PNG file.

    Author:
        Jakub Vašák
    """
    for sheet_name, dataframe in load_excel_sheets(file_path).items():
        try:
            processed_data = prepare_sheet_for_plotting(dataframe)
        except ValueError as error:
            print(f"Skipping sheet {sheet_name!r}: {error}")
            continue

        plot_excel_sheet(processed_data, sheet_name, save_plots=save_plots)

In [ ]:
def delete_and_create(report_path: Path) -> None:
    """Recreate an empty report directory.

    Args:
        report_path: Directory to remove and recreate.
    """
    with suppress(FileNotFoundError):
        shutil.rmtree(report_path)
    report_path.mkdir(parents=True, exist_ok=True)


def excel_to_df(path: Path, sheet: str) -> tuple[pd.DataFrame, str]:
    """Load one metrics sheet from an Excel workbook.

    Args:
        path: Path to the Excel workbook.
        sheet: Name of the sheet to load.

    Returns:
        Tuple containing the filtered metrics dataframe and workbook stem name.
    """
    name = path.stem
    df = pd.read_excel(path, sheet_name=sheet)
    df["Name"] = name
    columns = ["Name", "TP", "TN", "FP", "FN", "True", "False", "Accuracy"]
    return df[columns], name


def excel_list(excel_fold: Path) -> list[Path]:
    """List Excel workbooks in a directory.

    Args:
        excel_fold: Directory containing Excel files.

    Returns:
        Sorted list of ``.xls`` and ``.xlsx`` workbook paths.
    """
    return sorted(
        [path for path in excel_fold.iterdir() if path.suffix in {".xls", ".xlsx"}]
    )


def eva_into_csvs(excel_fold: Path, report_path: Path) -> None:
    """Convert selected evaluation sheets from Excel workbooks into CSV files.

    Args:
        excel_fold: Directory containing source evaluation workbooks.
        report_path: Directory where aggregated CSV files are written.

    Author:
        Jakub Vašák
    """
    sheets = [
        "ground truth",
        "accuracy +-1",
        "accuracy +-2",
        "accuracy after clean",
        "accuracy after clean +-1",
        "accuracy after clean +-2",
    ]
    for path in excel_list(excel_fold):
        for sheet in sheets:
            df, _ = excel_to_df(path, sheet)
            file_path = report_path / f"{sheet}.csv"
            header = ["Name", "TP", "TN", "FP", "FN", "True", "False", "Accuracy"]

            if not file_path.is_file():
                df.to_csv(file_path, index=False, header=header)
            else:
                df.to_csv(file_path, mode="a", index=False, header=False)


def excel_merge(report_path: Path, dest_path: Path) -> None:
    """Merge report CSV files into one multi-sheet Excel workbook.

    Args:
        report_path: Directory containing CSV files to merge.
        dest_path: Destination Excel workbook path.

    Author:
        Jakub Vašák
    """
    csv_files = sorted(
        [path for path in report_path.iterdir() if path.suffix == ".csv"]
    )

    with pd.ExcelWriter(dest_path, engine="openpyxl") as excel_writer:
        for csv_file in csv_files:
            df = pd.read_csv(csv_file)
            df.to_excel(excel_writer, sheet_name=csv_file.stem, index=False)


def excel_to_df_box(
    path: Path,
    sheet: str,
    row_index: int,
) -> tuple[pd.DataFrame, str]:
    """Load one box-level metrics row from an Excel workbook.

    Args:
        path: Path to the Excel workbook.
        sheet: Name of the sheet containing box-level metrics.
        row_index: Zero-based row index to extract from the sheet.

    Returns:
        Tuple containing the filtered box metrics dataframe and workbook stem name.
    """
    name = path.stem
    df = pd.read_excel(path, sheet_name=sheet)
    df["BOX_CONFUSION_MATRIX"] = name
    df = df.iloc[[row_index]]
    columns = ["BOX_CONFUSION_MATRIX", "TN", "TP", "FN", "FP", "acc"]
    return df[columns].rename(columns={"acc": "Accuracy"}), name


def eva_into_csvs_box(excel_fold: Path, report_path: Path) -> None:
    """Convert selected box-level evaluation rows into CSV files.

    Args:
        excel_fold: Directory containing source evaluation workbooks.
        report_path: Directory where aggregated box CSV files are written.

    Author:
        Jakub Vašák
    """
    row_mapping = {
        0: "box_clean_prob",
        1: "box_clean_prob_+-1",
        2: "box_clean_prob_+-2",
    }
    for path in excel_list(excel_fold):
        for row_index, row_name in row_mapping.items():
            df, _ = excel_to_df_box(path, "BOX", row_index)
            file_path = report_path / f"{row_name}.csv"
            header = ["Name", "TN", "TP", "FN", "FP", "Accuracy"]

            if not file_path.is_file():
                df.to_csv(file_path, index=False, header=header)
            else:
                df.to_csv(file_path, mode="a", index=False, header=False)

## Run the visualization

Run the next cell after the configuration and helper-function cells have been executed.

If `MERGED_REPORT_PATH` does not exist, first run the notebook/script that creates `MERGE_report.xlsx`, then return here and rerun the cells from the top.

In [ ]:
delete_and_create(REPORT_DIR)
eva_into_csvs(EVALUATION_DIR, REPORT_DIR)
eva_into_csvs_box(EVALUATION_DIR, REPORT_DIR)
excel_merge(REPORT_DIR, MERGED_REPORT_PATH)
plot_merged_excel_report(MERGED_REPORT_PATH, save_plots=False)